In [4]:
import pandas as pd
import numpy as np
import requests
import time

# 시각화 패키지
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 경고창 무시
import warnings
warnings.filterwarnings("ignore")

In [5]:
def load_riot_info(lan="ko_KR"):
    champ_ver = requests.get('https://ddragon.leagueoflegends.com/realms/na.json').json()['n']['champion']

    championJsonURL = f'http://ddragon.leagueoflegends.com/cdn/{champ_ver}/data/{lan}/champion.json'
    spellJsonURL = f'http://ddragon.leagueoflegends.com/cdn/{champ_ver}/data/{lan}/summoner.json'
    itemURL = f'http://ddragon.leagueoflegends.com/cdn/{champ_ver}/data/{lan}/item.json'

    # 데이터 프레임으로 만들기
    request1 = requests.get(championJsonURL)
    request2 = requests.get(spellJsonURL)
    request3 = requests.get(itemURL)

    champion_data = request1.json()
    champion_df = pd.DataFrame(champion_data['data']).T[["key","name"]]

    spell_data = request2.json()
    spell_df = pd.DataFrame(spell_data['data']).T[["key","name"]].reset_index(drop=True)

    item_data = request3.json()
    item_df = pd.DataFrame(item_data['data']).T[["name"]].reset_index()
    item_df.rename(columns={"index":"key"}, inplace=True)

    # key type 변경
    champion_df["key"] = champion_df["key"].astype(int)
    spell_df["key"] = spell_df["key"].astype(int)
    item_df["key"] = item_df["key"].astype(int)

    return champion_df, spell_df, item_df

In [6]:
champion_df, spell_df, item_df = load_riot_info(lan="ko_KR")

In [8]:
api_key = "RGAPI-516ddd05-257c-4712-96c2-5d2d929a91e2"

# 소환사 이름
summonerName = "hide on bush"
summoner_url = f"https://kr.api.riotgames.com/lol/summoner/v4/summoners/by-name/{summonerName}?api_key={api_key}"
summoner_r = requests.get(summoner_url)

# 나의 puuid
encryptedpuuid = summoner_r.json()['puuid']

KeyError: 'puuid'

In [ ]:
beginIndex = 0

# 매치 정보 (지수표현으로 나오지 않는지 확인)
my_matchid = np.array([], dtype=int)
d = 0
while True:
    match_url = f"https://asia.api.riotgames.com/lol/match/v5/matches/by-puuid/{encryptedpuuid}/ids?start={beginIndex}&count=20&api_key={api_key}"
    match_r = requests.get(match_url)
    print(match_r)
    # stop 조건: 불러온 json 파일의 length
    if len(pd.json_normalize(match_r.json())) == 0:
        break

    # 한번에 100의 매치 정보
    temp_matchid = match_r.json()
    temp_matchid
    my_matchid = np.concatenate([my_matchid, temp_matchid])

    # start index 변경
    beginIndex += 20
    d += 1
    if d >= 20000 :
      break
    # RATE LIMITS 피하기
    #time.sleep(1)

<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>


In [ ]:
my_matchid[0]

'KR_6823337832'

In [ ]:
my_game = pd.DataFrame()
d = 0
skip = 0
start_index = 0 #10명당 1게임

for i in my_matchid:
    if d >= 300:
      break
    gameid = i

    game_url = f"https://asia.api.riotgames.com/lol/match/v5/matches/{gameid}?api_key={api_key}"
    game_r = requests.get(game_url)
    print(game_r)
      # 10명의 대전 정보
    if game_r.json()['info']['gameMode'] == "CLASSIC" :
      game_info_participants = game_r.json()['info']['participants']
      temp_game = pd.json_normalize(game_info_participants)
      selected_fields = ['championName','item0','item1','item2','item3','item4','item5','item6','teamPosition','win']
      temp_game_selected = temp_game[selected_fields]
      temp_game_selected['matchId'] = [i,i,i,i,i,i,i,i,i,i]

      if game_r.json()['info']['gameDuration'] < 900:
        skip += 1
        continue
    # 나의 participantid 확인
      summoner_lst = pd.json_normalize(game_r.json()['info']['participants'])
      me = summoner_lst[summoner_lst['puuid'] == encryptedpuuid]

    # 대전 정보 stack

      my_game = my_game.append(temp_game_selected)
      d += 1
    # RATE LIMITS 피하기
      time.sleep(1)

<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200]>
<Response [200

In [ ]:
my_game

,championName,item0,item1,item2,item3,item4,item5,item6,teamPosition,win,matchId
0,Jayce,1036,1036,3070,1055,6691,2422,3340,TOP,False,KR_6823337832
1,Poppy,3158,1103,6632,0,0,1055,3340,JUNGLE,False,KR_6823337832
2,Orianna,6655,0,1052,1056,1082,3020,3363,MIDDLE,False,KR_6823337832
3,Varus,3153,1055,1042,3047,1042,2055,3340,BOTTOM,False,KR_6823337832
4,Braum,3855,3190,1029,3047,0,0,3364,UTILITY,False,KR_6823337832
...,...,...,...,...,...,...,...,...,...,...,...
5,Kayle,1028,0,0,3091,3006,0,3363,TOP,True,KR_6708937702
6,Viego,0,0,1103,3111,3078,6670,3513,JUNGLE,True,KR_6708937702
7,Azir,2031,4644,3108,3020,1056,1026,3340,MIDDLE,True,KR_6708937702
8,Xayah,1038,3133,6672,2422,1083,0,3363,BOTTOM,True,KR_6708937702


In [ ]:
my_game.to_csv(f"{summonerName}.csv")

In [ ]:
#my_game = pd.read_csv(f"Day02_02_hide on bush_log.csv")

In [ ]:
my_game[my_game['championName']=='Jayce']